# 47 — Closed-loop in silico (production codec + plant)

Encode a synthetic control reference into a spike raster, round-trip it through
the production lossless `SpikeCodec`, decode it, and drive a toy plant.
**Simulation only** — not a clinical BCI.

## Honesty box

| | |
|---|---|
| **Proves** | The production lossless codec preserves this seeded raster bit-exactly and its decoded signal drives a simulated feedback loop. |
| **Does not prove** | Implant hardware, stimulation safety, clinical efficacy, or lossy-codec performance. |
| **Models** | Production `SpikeCodec`; the plant is a discrete integrator. Perfect Integrator is a high-fidelity companion trace only. |


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from sc_neurocore.spike_codec.codec import SpikeCodec
from sc_neurocore.neurons.models import PerfectIntegratorNeuron

print("SC-NeuroCore — NB-47 closed-loop in silico")


In [ ]:
T = 200
t = np.arange(T)
reference = 0.5 + 0.4 * np.sin(2 * np.pi * t / 40.0)
rng = np.random.default_rng(0)
source_raster = (rng.random(T) < np.clip(reference, 0, 1)).astype(np.int8).reshape(T, 1)

codec = SpikeCodec(mode="lossless", entropy="varint")
payload, codec_stats = codec.compress(source_raster)
restored_raster = codec.decompress(payload, T=T, N=1)
codec_round_trip_exact = bool(np.array_equal(restored_raster, source_raster))
assert codec_round_trip_exact, "lossless SpikeCodec changed the source raster"

decoded = np.zeros(T)
acc = 0.0
for i, spike in enumerate(restored_raster[:, 0].astype(float)):
    acc = 0.9 * acc + spike
    decoded[i] = acc
decoded = decoded / max(float(decoded.max()), 1e-9)

# Toy plant: integrate error from the decoded production-codec signal.
plant = np.zeros(T)
u = 0.0
for i in range(1, T):
    err = decoded[i - 1] - plant[i - 1]
    u = 0.6 * u + 0.4 * err
    plant[i] = np.clip(plant[i - 1] + 0.15 * u, 0, 1)
tracking_mae = float(np.mean(np.abs(plant - decoded)))

pi = PerfectIntegratorNeuron()
_v, pi_spikes = pi.simulate(T, current=1.5)

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
axes[0].plot(t, reference, label="synthetic reference")
axes[0].plot(t, decoded, label="decoded restored raster", alpha=0.8)
axes[0].legend(); axes[0].set_ylabel("signal"); axes[0].grid(True, alpha=0.3)
axes[1].plot(t, plant, color="C2"); axes[1].set_ylabel("plant"); axes[1].grid(True, alpha=0.3)
axes[2].plot(t, _v, color="C3"); axes[2].set_ylabel("PI v"); axes[2].set_xlabel("step")
axes[2].set_title(f"Perfect Integrator companion spikes={pi_spikes}")
axes[2].grid(True, alpha=0.3)
fig.suptitle("Closed-loop in silico (not clinical)")
fig.tight_layout()
plt.show()
print(codec_stats.summary())
print(f"bit-exact round trip={codec_round_trip_exact}, tracking MAE={tracking_mae:.4f}")
print("NB-47 complete: production codec path, no fallback.")
